<a href="https://colab.research.google.com/github/dkdkndmd/nanobanana/blob/main/ComfyUIonColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

# `Models Download`

## Download MODELS

In [ ]:
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2


from google.colab import userdata

CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
print("Loaded API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")

# Install custom nodes

def install_custom_node(url):
  %cd /content/ComfyUI/custom_nodes
  !git clone {url}

def downloadModel(url, filename = None):
  if 'huggingface.co' in url:
    if filename is None:
      filename = url.split('/')[-1]
      filename = filename.removesuffix('?download=true')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}  -o {filename}
  else:
    # civitai
    if filename:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN} -o {filename}
    else:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN}


# install custom nodes
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Impact-Pack')


# download models
%cd ./models/checkpoints

# CyberRealistic Pony
downloadModel('https://civitai.com/api/download/models/2071650')

# Protovision XL
#downloadModel('https://civitai.com/api/download/models/201514')
# Juggernaut XL Hyper
#downloadModel('https://civitai.com/api/download/models/471120')
# Juppernaut X
#downloadModel('https://civitai.com/api/download/models/456194')
# Juggernaut v8
#downloadModel('https://civitai.com/api/download/models/288982')

# Huggingface example
#downloadModel('https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors?download=true')

%cd {WORKSPACE}

## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [ ]:
!ls -al ./models/checkpoints/

# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
%cd /content/ComfyUI

print("="*50)
print("🚀 启动 ComfyUI 并下载模型")
print("="*50)

# ========== 1. 启动 ComfyUI ==========
print("\n1️⃣ 启动 ComfyUI...")
import subprocess, time, socket, os
os.system("pkill -f main.py 2>/dev/null")
os.system("nohup python main.py --dont-print-server > comfyui.log 2>&1 &")

# 等待启动
for i in range(30):
    time.sleep(1)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', 8188))
    sock.close()
    if result == 0:
        print("   ✅ ComfyUI 已启动 (端口 8188 已监听)")
        break
else:
    print("   ❌ ComfyUI 启动超时")
    !tail -20 comfyui.log

# ========== 2. 下载模型文件 ==========
print("\n2️⃣ 下载模型文件...")

# 创建目录
os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/loras", exist_ok=True)

# 主模型
if not os.path.exists("models/checkpoints/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf"):
    print("   ⬇️ 下载主模型 (1.5GB)...")
    os.system('wget -q --show-progress -O models/checkpoints/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf "https://huggingface.co/kekusprod/WAI-NSFW-illustrious-SDXL-v110-GGUF/resolve/main/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf"')
else:
    print("   ✅ 主模型已存在")

# CLIP G
if not os.path.exists("models/text_encoders/illustrious_clip_g_fp8_e4m3fn.safetensors"):
    print("   ⬇️ 下载 CLIP G...")
    os.system('wget -q --show-progress -O models/text_encoders/illustrious_clip_g_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_g_fp8_e4m3fn.safetensors"')
else:
    print("   ✅ CLIP G 已存在")

# CLIP L
if not os.path.exists("models/text_encoders/illustrious_clip_l_fp8_e4m3fn.safetensors"):
    print("   ⬇️ 下载 CLIP L...")
    os.system('wget -q --show-progress -O models/text_encoders/illustrious_clip_l_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_l_fp8_e4m3fn.safetensors"')
else:
    print("   ✅ CLIP L 已存在")

# VAE
if not os.path.exists("models/vae/illustrious_v110_vae_fp8_e4m3fn.safetensors"):
    print("   ⬇️ 下载 VAE...")
    os.system('wget -q --show-progress -O models/vae/illustrious_v110_vae_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_v110_vae_fp8_e4m3fn.safetensors"')
else:
    print("   ✅ VAE 已存在")

# LoRA（从 Civitai）
if not os.path.exists("models/loras/etuzan_jakusui_illustrious.safetensors"):
    print("   ⬇️ 下载越山弱衰 LoRA...")
    os.system('wget -q --show-progress --header="User-Agent: Mozilla/5.0" -O models/loras/etuzan_jakusui_illustrious.safetensors "https://civitai.com/api/download/models/1590157?fileId=1490031"')
else:
    print("   ✅ LoRA 已存在")

# ========== 3. 验证文件 ==========
print("\n3️⃣ 验证文件...")
!ls -lh models/checkpoints/ models/text_encoders/ models/vae/ models/loras/ 2>/dev/null

# ========== 4. 启动 Cloudflare Tunnel ==========
print("\n4️⃣ 启动 Cloudflare Tunnel...")
!cloudflared tunnel --url http://127.0.0.1:8188

/content/ComfyUI
🚀 启动 ComfyUI 并下载模型

1️⃣ 启动 ComfyUI...
   ✅ ComfyUI 已启动 (端口 8188 已监听)

2️⃣ 下载模型文件...
   ⬇️ 下载主模型 (1.5GB)...
   ⬇️ 下载 CLIP G...
   ⬇️ 下载 CLIP L...
   ⬇️ 下载 VAE...
   ⬇️ 下载越山弱衰 LoRA...

3️⃣ 验证文件...
models/checkpoints/:
total 1.4G
-rw-r--r-- 1 root root    0 Jul 15 09:33 put_checkpoints_here
-rw-r--r-- 1 root root 1.4G Jul 15 09:47 WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf

models/loras/:
total 12K
-rw-r--r-- 1 root root 9.5K Jul 15 09:47 etuzan_jakusui_illustrious.safetensors
-rw-r--r-- 1 root root    0 Jul 15 09:33 put_loras_here

models/text_encoders/:
total 781M
-rw-r--r-- 1 root root 663M Jul 15 09:47 illustrious_clip_g_fp8_e4m3fn.safetensors
-rw-r--r-- 1 root root 118M Jul 15 09:47 illustrious_clip_l_fp8_e4m3fn.safetensors
-rw-r--r-- 1 root root    0 Jul 15 09:33 put_text_encoder_files_here

models/vae/:
total 80M
-rw-r--r-- 1 root root 80M Jul 15 09:47 illustrious_v110_vae_fp8_e4m3fn.safetensors
-rw-r--r-- 1 root root   0 Jul 15 09:33 put_vae_here

4️⃣ 启动 Cloudflar

In [ ]:
%cd /content/ComfyUI

# 1. 停止 Cloudflare
!pkill -f cloudflared

# 2. 安装 localtunnel（如果未安装）
!npm install -g localtunnel

# 3. 启动 localtunnel（会输出新链接）
!lt --port 8188